# 4. Rating Prediction (Regression)

In [3]:
import re
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
def fix_negations(text):
    text = str(text).lower()
    return re.sub(
        r"\b(not|no|never|without|don't|doesn't|didn't|isn't|aren't|wasn't|weren't|won't|wouldn't|couldn't|can't)\s+(\w+)",
        r"\1_\2", text)

df = pd.read_csv("../data/processed/cleaned_restaurant_reviews.csv")
X_text = df["review_clean"].astype(str).apply(fix_negations)
y = df["Rating"]
print("Reviews:", len(df))

Reviews: 587


### Load the saved TF-IDF vectorizer

In [5]:
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")
print("Vocabulary size:", len(tfidf.get_feature_names_out()))

Vocabulary size: 4230


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

rating_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vec_new = rating_tfidf.fit_transform(X_train)
X_test_vec_new = rating_tfidf.transform(X_test)

print("New vocabulary size:", len(rating_tfidf.get_feature_names_out()))
print("Train shape:", X_train_vec_new.shape)
print("Test shape:", X_test_vec_new.shape)

New vocabulary size: 1503
Train shape: (469, 1503)
Test shape: (118, 1503)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)

X_train_vec = tfidf.transform(X_train)
X_test_vec  = tfidf.transform(X_test)
print(X_train_vec.shape, X_test_vec.shape)

(469, 4230) (118, 4230)


### Train the regressor

In [ ]:
reg = Ridge(alpha=1.0, random_state=42)
reg.fit(X_train_vec, y_train)
 
pred = reg.predict(X_test_vec)
print("MSE:", round(mean_squared_error(y_test, pred), 3))
print("R2:", round(r2_score(y_test, pred), 3))
print("Predicted range on test set:", round(pred.min(), 2), "-", round(pred.max(), 2))

MSE: 0.997
R2: 0.389
Predicted range on test set: 2.22 - 5.08


In [10]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

reg_new = Ridge(alpha=1.0)

reg_new.fit(X_train_vec_new, y_train)

pred_new = reg_new.predict(X_test_vec_new)

mse_new = mean_squared_error(y_test, pred_new)
rmse_new = np.sqrt(mse_new)
r2_new = r2_score(y_test, pred_new)

print("New MSE:", round(mse_new, 3))
print("New RMSE:", round(rmse_new, 3))
print("New R2:", round(r2_new, 3))
print(
    "Predicted range:",
    round(pred_new.min(), 2),
    "-",
    round(pred_new.max(), 2)
)

New MSE: 0.911
New RMSE: 0.954
New R2: 0.442
Predicted range: 2.22 - 5.2


In [11]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

alphas = [0.01, 0.1, 0.5, 1, 2, 5, 10]

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_vec_new, y_train)

    pred = model.predict(X_test_vec_new)

    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    print(
        f"Alpha: {alpha:<5} | "
        f"MSE: {mse:.3f} | "
        f"RMSE: {np.sqrt(mse):.3f} | "
        f"R2: {r2:.3f}"
    )

Alpha: 0.01  | MSE: 1.283 | RMSE: 1.133 | R2: 0.214
Alpha: 0.1   | MSE: 1.098 | RMSE: 1.048 | R2: 0.327
Alpha: 0.5   | MSE: 0.926 | RMSE: 0.962 | R2: 0.433
Alpha: 1     | MSE: 0.911 | RMSE: 0.954 | R2: 0.442
Alpha: 2     | MSE: 0.962 | RMSE: 0.981 | R2: 0.411
Alpha: 5     | MSE: 1.126 | RMSE: 1.061 | R2: 0.310
Alpha: 10    | MSE: 1.281 | RMSE: 1.132 | R2: 0.215


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_vec_new, y_train)

rf_pred = rf.predict(X_test_vec_new)

rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MSE:", round(rf_mse, 3))
print("Random Forest RMSE:", round(rf_rmse, 3))
print("Random Forest R2:", round(rf_r2, 3))

Random Forest MSE: 0.821
Random Forest RMSE: 0.906
Random Forest R2: 0.497


In [13]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

extra = ExtraTreesRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

extra.fit(X_train_vec_new, y_train)

extra_pred = extra.predict(X_test_vec_new)

extra_mse = mean_squared_error(y_test, extra_pred)
extra_rmse = np.sqrt(extra_mse)
extra_r2 = r2_score(y_test, extra_pred)

print("Extra Trees MSE:", round(extra_mse, 3))
print("Extra Trees RMSE:", round(extra_rmse, 3))
print("Extra Trees R2:", round(extra_r2, 3))

Extra Trees MSE: 0.786
Extra Trees RMSE: 0.887
Extra Trees R2: 0.518


In [14]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Build the complete model pipeline
cv_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("model", ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = cross_val_score(
    cv_model,
    X_text,
    y,
    cv=kf,
    scoring="r2"
)

rmse_scores = -cross_val_score(
    cv_model,
    X_text,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

print("R2 scores:", np.round(r2_scores, 3))
print("Average R2:", round(r2_scores.mean(), 3))
print("Average RMSE:", round(rmse_scores.mean(), 3))


R2 scores: [0.518 0.439 0.275 0.433 0.395]
Average R2: 0.412
Average RMSE: 1.005


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# Same folds for every model
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Ridge": Ridge(alpha=1.0),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("model", model)
    ])

    r2_scores = cross_val_score(
        pipeline,
        X_text,
        y,
        cv=kf,
        scoring="r2"
    )

    rmse_scores = -cross_val_score(
        pipeline,
        X_text,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    print("\n", name)
    print("R2 scores:", np.round(r2_scores, 3))
    print("Average R2:", round(r2_scores.mean(), 3))
    print("Average RMSE:", round(rmse_scores.mean(), 3))


 Ridge
R2 scores: [0.442 0.429 0.519 0.36  0.407]
Average R2: 0.431
Average RMSE: 0.991

 Random Forest
R2 scores: [0.501 0.389 0.396 0.423 0.345]
Average R2: 0.411
Average RMSE: 1.008

 Extra Trees
R2 scores: [0.518 0.439 0.275 0.433 0.395]
Average R2: 0.412
Average RMSE: 1.005


In [16]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

# Final TF-IDF vectorizer
final_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit TF-IDF on all review text
X_final = final_tfidf.fit_transform(X_text)

# Final Ridge model
final_rating_model = Ridge(alpha=1.0)
final_rating_model.fit(X_final, y)

# Save BOTH together
joblib.dump(final_tfidf, "../models/rating_tfidf_vectorizer.pkl")
joblib.dump(final_rating_model, "../models/rating_ridge_model.pkl")

print("Final rating model saved successfully!")
print("Vocabulary size:", len(final_tfidf.get_feature_names_out()))

Final rating model saved successfully!
Vocabulary size: 1790


### Compare predictors vs actual ratings

In [ ]:
comparison = pd.DataFrame({"actual": y_test.values,"predicted": pred.round(1)}).reset_index(drop=True)
comparison.head(15)

,actual,predicted
0,4.0,4.2
1,5.0,4.4
2,5.0,4.6
3,4.0,4.3
4,4.0,4.0
5,5.0,4.4
6,4.0,3.7
7,5.0,4.6
8,5.0,3.8
9,5.0,4.9


### Save the model

In [ ]:
joblib.dump(reg, "../models/rating_regressor.pkl")
print("Saved.")

Saved.


In [ ]:
import os
import joblib
from sklearn.linear_model import Ridge

# Train final Ridge model using the NEW rating TF-IDF features
final_rating_model = Ridge(alpha=1.0)
final_rating_model.fit(X_train_vec_new, y_train)

# Make sure models folder exists
os.makedirs("../models", exist_ok=True)

# Save them separately from the sentiment TF-IDF
joblib.dump(final_rating_model, "../models/rating_regressor.pkl")
joblib.dump(rating_tfidf, "../models/rating_tfidf_vectorizer.pkl")

print("Rating model saved successfully!")
print("Saved: rating_regressor.pkl")
print("Saved: rating_tfidf_vectorizer.pkl")

Rating model saved successfully!
Saved: rating_regressor.pkl
Saved: rating_tfidf_vectorizer.pkl


: 